# Five-year exit research: a visual **diagnostic**, not a trading decision

This walk-through uses the **private** 2020–2024 OANDA practice minute quotes and the previously frozen nine exit rules. All fills and dollar differences below are **hypothetical**, not broker trades or account balances. The original `NSXUSD` CSVs are absent and earlier historical notebooks were inspected: **none of these years is an untouched holdout**. There is no winner and the live 25/75 bot is unchanged.

**Safety:** This tracked template is intentionally UNEXECUTED. Open only an ignored `notebooks/07_historical_rolling_diagnostics.local.ipynb` copy (instructions in `notebooks/README.md`). Executed charts are private. Do not commit or publicly share them. No OANDA/Cloud/X request is made by this notebook.

## In human terms: why the colored blocks and what-if charts?

We asked the same nine closing-rule questions for every day with reliable quotes. **Development 2020–21 → check 2022; 2020–22 → check 2023; 2020–23 → check 2024.** This chronology is a useful retrospective stability check, but it does **not** become a future test because old historical results were already viewed. A day with missing quotes or uncertain within-minute order is removed from **all** nine rules and both cost cases; true no-trade days stay as zero. The chart of those exclusions is as important as the performance charts.

The two price scenarios include actual **quoted** bid/ask spread. The extra-cost scenario adds a *hypothetical* 1 point at entry and exit plus $2 per fill; those charges are **not** verified historical OANDA fees. Profit-protection, trailing and partial fills cannot be certified by minute candles.

## 0. Offline load and freeze check

The loader re-audits the **local** quote-batch checksums, private historical baseline, both committed protocols and simulator source fingerprints against the report. It can take several seconds to read the ignored raw quote files; it never prints those quotes or accesses `.env`, broker transactions or the OANDA API. If anything differs, review and regenerate the private study before viewing any figure.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import yaml

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
path = ROOT / 'data/raw/oanda_research/history/rolling_exit_study.json'
if not path.is_file():
    raise FileNotFoundError('Run the OFFLINE scripts/research/run_historical_exit_study.py after the five-year quote audit.')
raw_candidates = (ROOT / 'research/experiments.yml').read_bytes()
raw_folds = (ROOT / 'research/rolling-origin.yml').read_bytes()
protocol = yaml.safe_load(raw_candidates)
fold_protocol = yaml.safe_load(raw_folds)
import sys
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.research_rolling import validate_protocol
from src.research_visuals import LABELS
from scripts.research.run_historical_exit_study import verify_private_report_freshness
validate_protocol(protocol, fold_protocol)
report = json.loads(path.read_text(encoding='utf-8'))
verify_private_report_freshness(report)  # Local-only source, baseline, code and protocol SHA-256 checks.
ORDER = report['candidate_order']  # Original predeclared order; NEVER sort by outcomes.
FOLD_IDS = [x['id'] for x in fold_protocol['folds']]
COST_CASE = 'adverse_extra_1pt_per_fill'  # Or 'bid_ask_only'; both already include quote spread.
if COST_CASE not in report['cases']:
    raise ValueError('Unknown declared cost scenario')
case = report['cases'][COST_CASE]
print('Private retrospective diagnostics | years: 2020–2024 | paired days:', len(report['common_eligible_dates']))
print('Uncertain minute order exclusions:', report['exclusion_reason_counts'].get('one_minute_ordering_ambiguous', 0))
print('Never an untouched holdout; no rule selected.')

## 1. How many days do the pictures really represent?

Green = modelled trade signal on a **clean** paired day; light green = genuinely no-trade signal retained as zero; orange/red = missing quotes or uncertain ordering for **at least one** rule/cost. A large red region is not a string of losing trades: it means we **cannot safely compare** those days with one-minute prices. Omitting volatile days could bias the clean sample.

In [ ]:
years = list(report['coverage_by_year'])
categories = [('baseline_trade_days', 'Clean trade signal', '#167d78'),
              ('no_trade_days', 'Clean no-trade signal', '#88cbb8'),
              ('excluded_quote_days', 'Missing/incomplete quotes', '#d38319'),
              ('excluded_ordering_or_simulator_days', 'Uncertain minute ordering', '#a43952')]
fig, ax = plt.subplots(figsize=(10, 4.4), layout='constrained')
bottom = np.zeros(len(years))
for key, label, color in categories:
    values = np.array([report['coverage_by_year'][y][key] for y in years])
    ax.bar(years, values, bottom=bottom, label=label, color=color)
    bottom += values
ax.set_ylabel('Requested NY weekdays')
ax.set_title('All requested days, including exclusions and real no-trade signals')
ax.legend(loc='lower center', bbox_to_anchor=(.5, -0.32), ncol=2, frameon=False)
plt.show()

## 2. Fixed-order comparison: how stable is the *hypothetical* daily difference?

Each square is candidate **minus the current 25/75 rule on the same eligible day**, averaged over a retrospective check year. Blue/red means more/less modelled net dollars **in this historical feed only**. Rows remain in the frozen order, **not a leaderboard**. Both cost panels use the same days and color scale. Dollar-per-point, real historical commissions and counterfactual fills are unverified; do **not** select the bluest row.

In [ ]:
cost_ids = report['cost_case_order']
values = []
for cost_id in cost_ids:
    folds = report['cases'][cost_id]['folds']
    values.append(np.array([[next(x for x in folds[fold]['evaluation_in_fixed_policy_order'] if x['id'] == pid)
                             ['paired_mean_net_usd_per_session'] for fold in FOLD_IDS] for pid in ORDER]))
limit = max(1., *(np.abs(v).max() for v in values))
fig, axes = plt.subplots(1, 2, figsize=(13, 5.4), sharey=True, layout='constrained')
for ax, cost_id, v in zip(axes, cost_ids, values):
    image = ax.imshow(v, cmap='RdBu', vmin=-limit, vmax=limit, aspect='auto')
    ax.set_xticks(range(3), ['2022', '2023', '2024'])
    ax.set_yticks(range(len(ORDER)), [LABELS[x] for x in ORDER], fontsize=8)
    ax.set_title(cost_id.replace('_', ' ') + '\nRetrospective check years; NOT untouched')
fig.colorbar(image, ax=axes, shrink=.8, label='Hypothetical paired mean net USD / eligible day')
plt.show()

## 3. Did omitting the uncertain days change the picture? **Post-hoc sensitivity**

After seeing how many days had ambiguous minute ordering, we separately computed the SAME model with flagged days included. **Left** is the predeclared clean sample; **right** includes flagged days under the assumed stop-first/deferred-amendment sequence. Right is **not** a proven worst case, a broker fill bound or a replacement sample. A color reversal means execution data, not optimism, must resolve the question. No bootstrap probabilities are quoted for this post-hoc sensitivity.

In [ ]:
years = ['2022', '2023', '2024']
clean = np.array([[case['by_year_in_fixed_policy_order'][y][pid]['paired_mean_net_usd_per_session']
                   for y in years] for pid in ORDER])
inclusive = np.array([[next(x for x in case['posthoc_ambiguity_inclusive_descriptive_by_year_in_fixed_policy_order'][y]
                            if x['id'] == pid)['paired_mean_net_usd_per_session']
                       for y in years] for pid in ORDER])
limit = max(1., np.abs(clean).max(), np.abs(inclusive).max())
fig, axes = plt.subplots(1, 2, figsize=(13, 5.3), sharey=True, layout='constrained')
for ax, matrix, title in zip(axes, [clean, inclusive],
                             ['Predeclared clean paired days', 'POST-HOC modelled ambiguity included']):
    image = ax.imshow(matrix, cmap='RdBu', vmin=-limit, vmax=limit, aspect='auto')
    ax.set_xticks(range(3), years)
    ax.set_yticks(range(len(ORDER)), [LABELS[x] for x in ORDER], fontsize=8)
    ax.set_title(title + '\n' + COST_CASE.replace('_', ' '))
fig.colorbar(image, ax=axes, shrink=.8, label='Hypothetical paired mean net USD / modelled day')
plt.show()

## 4. Cost reality check: quotes, not actual execution bills

Historical 10:23 **quoted** ask minus bid changes over time. This is already in both scenarios. Median and 95th-percentile quoted spread do **not** cover extra slippage, commission, partial-fill mechanics or contract changes. The plotted values are index points, NOT account balances.

In [ ]:
spread = report['quoted_entry_spread_points_by_year']
years = list(spread)
fig, ax = plt.subplots(figsize=(9, 3.5), layout='constrained')
ax.plot(years, [spread[y]['median_points'] for y in years], marker='o', label='Median quoted spread')
ax.plot(years, [spread[y]['p95_points'] for y in years], marker='s', label='95th-percentile quoted spread')
ax.set_ylabel('OANDA NAS100_USD quote points')
ax.set_title('Observed 10:23 historical quotes — NOT a full execution cost model')
ax.legend(frameon=False)
plt.show()

## 5. Why uncertainty bands still do NOT settle the question

The final retrospective check year clears the **26 calendar weeks / 100 baseline trade days** gate, so paired week-block resampling yields *descriptive* 5–95% intervals for the predeclared CLEAN sample. These are **not** multiple-testing-adjusted, do not repair skipped volatile days or minute OHLC fill uncertainty, and do not predict future profit. If the gate ever fails, no intervals are drawn. Keep prospective future practice data genuinely unseen until a candidate is frozen.

In [ ]:
fold = case['folds']['through_2023_check_2024']
band = fold['evaluation_bootstrap_in_fixed_policy_order']
fig, ax = plt.subplots(figsize=(10, 5), layout='constrained')
if all(x['status'] == 'descriptive_bootstrap_only' for x in band):
    for i, item in enumerate(band):
        low, mid, high = item['paired_mean_usd_per_session_p05_p50_p95']
        ax.plot([low, high], [i, i], color='#167d78', lw=2)
        ax.scatter(mid, i, color='#134653', zorder=3)
    ax.axvline(0, color='black', lw=.8)
    ax.set_yticks(range(len(ORDER)), [LABELS[x] for x in ORDER], fontsize=9)
    ax.invert_yaxis()
    ax.set_xlabel('Hypothetical paired net USD / day; bootstrap 5–95%, UNADJUSTED')
    ax.set_title('2024 retrospective week resampling — NOT a future profit claim')
else:
    ax.text(.5, .5, 'INSUFFICIENT HISTORY — no intervals', ha='center', va='center', transform=ax.transAxes)
    ax.set_axis_off()
plt.show()

## What we must learn before changing any trading rule

Investigate the **11 incomplete and 14 quote-free weekdays**, the **151 ambiguous trade days**, historical CFD contract/point value and true order costs. Check whether an apparent comparison flips when ambiguous days are included under the model. More importantly, get finer execution evidence for partials/protected stops and keep **genuinely new practice sessions** for forward confirmation. No colorful square here authorizes a live bot change.